# DG-MCTS · Reproduce the numerical results

Recompute Tables 2–6 and reward-parameter sensitivity from the released observation-level data. This notebook runs offline and does not call an LLM or optimization solver.

In [1]:
import sys
from pathlib import Path
sys.dont_write_bytecode = True
import pandas as pd
from IPython.display import display
root = Path.cwd()
if not (root / 'code/reproduce.py').exists():
    raise RuntimeError('Open this notebook from the repository root.')
sys.path.insert(0, str(root / 'code'))
import reproduce
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 30)
tables = reproduce.reproduce()
print('Tables recomputed from observations; eight CSV files saved in results/.')

Tables recomputed from observations; eight CSV files saved in results/.


## Table 2 · Core solving accuracy

Six benchmarks and four DG-MCTS backbones. Reference-only cells are read as declared aggregate comparisons; other cells are calculated from instance outcomes.

In [2]:
display(pd.DataFrame(tables['table2_core_accuracy']))

,method,NL4Opt,NLP4LP,Mamo-E,ComplexOR,Mamo-C,IndustryOR
0,IO (GPT-4o-mini),72.3%,72.7%,83.1%,55.6%,33.7%,35.0%
1,IO (Gemini-2.5),77.5%,73.6%,81.6%,72.2%,47.4%,43.0%
2,IO (o4-mini),77.9%,81.8%,83.7%,72.2%,43.6%,50.0%
3,IO (GPT-5.4-mini),80.5%,83.1%,86.8%,77.8%,66.4%,56.0%
4,CoE (Gemini-2.5),82.3%,84.7%,84.5%,83.3%,65.4%,58.0%
5,CoE (GPT-5.4-mini),84.0%,85.5%,86.8%,83.3%,66.4%,51.0%
6,OptiMUS (Gemini-2.5),87.0%,84.3%,83.3%,61.1%,68.7%,54.0%
7,OptiMUS (GPT-5.4-mini),86.6%,85.1%,87.1%,83.3%,64.5%,53.0%
8,ECO (GPT-4o-mini),79.2%,86.8%,86.7%,77.8%,57.4%,43.0%
9,ECO (o4-mini),86.2%,86.8%,84.4%,83.3%,71.6%,59.0%


## Table 3 · Extended benchmark suite

The same named OptiBench targets are applied to both inference methods. Numerical agreement is an evaluation criterion, not a correctness certificate for every modeling constraint.

In [3]:
display(pd.DataFrame(tables['table3_extended_accuracy']))

,benchmark,reference_method,published_accuracy,cases,fast_only_correct,fast_only_accuracy_percent,dg_mcts_correct,dg_mcts_accuracy_percent
0,Large-Scale-OR,LEAN-LLM-OPT,85.1%,101,94,93.1,95,94.1
1,Air-NRM (CA),LEAN-LLM-OPT,93.3%,15,13,86.7,15,100.0
2,Air-NRM (NP-Flow),LEAN-LLM-OPT,92.1%,21,18,85.7,20,95.2
3,OptiBench,LLMOPT (Qwen2.5-14B),66.4%,605,443,73.2,464,76.7
4,OptMATH-Bench,LLMOPT (Qwen2.5-14B),40.0%,166,61,36.7,70,42.2


## Table 4 · Inference routes

Accuracy and computational costs are aggregated from their respective released records. The overall costs weight each instance equally.

In [4]:
routes = pd.DataFrame(tables['table4_routes'])
display(routes[routes.dataset == 'Overall'].reset_index(drop=True))
display(routes[routes.dataset != 'Overall'].reset_index(drop=True))

,dataset,method,cases,correct,accuracy_percent,mean_llm_calls,mean_tokens,mean_wall_seconds
0,Overall,Fast Only,250,177,70.8,3.2,13192,19.3
1,Overall,DG-MCTS,250,190,76.0,62.9,259324,305.3
2,Overall,Always Slow,250,189,75.6,96.8,374658,403.4


,dataset,method,cases,correct,accuracy_percent,mean_llm_calls,mean_tokens,mean_wall_seconds
0,Mamo-E sample,Fast Only,50,47,94.0,3.3,11185,17.4
1,Mamo-E sample,DG-MCTS,50,46,92.0,40.0,130006,142.9
2,Mamo-E sample,Always Slow,50,45,90.0,98.6,303280,348.9
3,Mamo-C sample,Fast Only,50,36,72.0,3.2,16456,22.4
4,Mamo-C sample,DG-MCTS,50,38,76.0,48.2,253578,233.2
5,Mamo-C sample,Always Slow,50,37,74.0,97.0,445617,419.6
6,NLP4LP sample,Fast Only,50,43,86.0,3.1,10309,14.8
7,NLP4LP sample,DG-MCTS,50,44,88.0,70.9,219600,269.4
8,NLP4LP sample,Always Slow,50,45,90.0,87.4,267023,361.2
9,IndustryOR easy,Fast Only,40,26,65.0,3.2,12728,17.7


## Table 5 · Acceptance and final correctness

Use saved final-correctness labels for the routing analysis. Acceptance coverage uses all instances in the group; route accuracies use their respective accepted or slow-search denominators.

In [5]:
display(pd.DataFrame(tables['table5_routing']))

,dataset,cases,fast_accepted,acceptance_coverage_percent,slow_entered,slow_coverage_percent,accepted_correct,accepted_accuracy_percent,false_acceptances,slow_correct,slow_accuracy_percent,final_correct,final_accuracy_percent
0,ComplexOR,18,6,33.3,12,66.7,4,66.7,2,11,91.7,15,83.3
1,IndustryOR easy,40,9,22.5,31,77.5,8,88.9,1,23,74.2,31,77.5
2,IndustryOR medium,40,2,5.0,38,95.0,2,100.0,0,17,44.7,19,47.5
3,IndustryOR hard,20,2,10.0,18,90.0,2,100.0,0,10,55.6,12,60.0
4,Mamo-E,652,451,69.2,201,30.8,414,91.8,37,162,80.6,576,88.3
5,Mamo-C,211,82,38.9,129,61.1,63,76.8,19,93,72.1,156,73.9
6,NL4Opt,231,47,20.3,184,79.7,43,91.5,4,162,88.0,205,88.7
7,NLP4LP,242,71,29.3,171,70.7,66,93.0,5,144,84.2,210,86.8
8,Overall,1454,670,46.1,784,53.9,602,89.9,68,622,79.3,1224,84.2


## Table 6 · Formulation reuse and diagnoses

Formulation-origin shares are conditional on a correct slow-search outcome. Diagnostic categories are recomputed from structured labels using the released mapping; one instance can belong to multiple categories.

In [6]:
display(pd.DataFrame(tables['table6_search_outcomes']))
display(pd.DataFrame(tables['table6_diagnosis']))

,dataset,slow_entered,final_correct,initial_reused,initial_share_percent,revised_selected,revised_share_percent
0,ComplexOR,12,11,2,18.2,9,81.8
1,IndustryOR easy,31,23,13,56.5,10,43.5
2,IndustryOR medium,38,17,6,35.3,11,64.7
3,IndustryOR hard,18,10,3,30.0,7,70.0
4,Mamo-E,201,162,22,13.6,140,86.4
5,Mamo-C,129,93,22,23.7,71,76.3
6,NL4Opt,184,162,60,37.0,102,63.0
7,NLP4LP,171,144,52,36.1,92,63.9
8,Overall,784,622,180,28.9,442,71.1


,category,detected,final_correct,final_incorrect,recovery_percent
0,Answer/reporting error,421,337,84,80.0
1,Code/execution error,120,79,41,65.8
2,Solver status / feasibility error,41,22,19,53.7
3,Variable/domain error,103,68,35,66.0
4,Objective modeling error,112,73,39,65.2
5,Constraint modeling error,130,82,48,63.1
6,Parameter / numeric coverage error,334,231,103,69.2
7,Unit / business semantics error,89,59,30,66.3


## Reward weights and risk caps

The parameter file preserves the five automatic-score weights, six semantic-review weights, and all three risk-cap settings. Each variant is scored on its 150 intended targets. One implementation-heavy target lacks a final record and is scored unsuccessful under the declared post-hoc convention; its observed outcome remains missing. Baseline comparisons are descriptive, using the reported aggregate rather than a recovered paired baseline. Percentage-point differences follow the paper’s displayed one-decimal percentages.

In [7]:
display(pd.DataFrame(tables['reward_sensitivity']))
display(pd.DataFrame(tables['reward_by_dataset']))

,variant,dataset,target_cases,observed_final_results,correct,accuracy_percent,baseline_correct,baseline_cases,baseline_accuracy_percent,difference_pp
0,rule_execution_heavy,Overall,150,150,97,64.7,103,150,68.7,-4.0
1,rule_objective_heavy,Overall,150,150,97,64.7,103,150,68.7,-4.0
2,rule_structure_heavy,Overall,150,150,97,64.7,103,150,68.7,-4.0
3,sem_data_definition_heavy,Overall,150,150,99,66.0,103,150,68.7,-2.7
4,sem_implementation_heavy,Overall,150,149,94,62.7,103,150,68.7,-6.0
5,sem_math_structure_heavy,Overall,150,150,97,64.7,103,150,68.7,-4.0
6,cap_core_model_heavy,Overall,150,150,103,68.7,103,150,68.7,0.0
7,cap_revision_heavy,Overall,150,150,97,64.7,103,150,68.7,-4.0
8,cap_semantic_numeric_heavy,Overall,150,150,98,65.3,103,150,68.7,-3.4


,variant,dataset,target_cases,observed_final_results,correct,accuracy_percent,baseline_correct,baseline_cases,baseline_accuracy_percent,difference_pp
0,rule_execution_heavy,mamo_easy,40,40,36,90.0,36,40,90.0,0.0
1,rule_execution_heavy,mamo_complex,50,50,34,68.0,38,50,76.0,-8.0
2,rule_execution_heavy,industryor_medium,40,40,16,40.0,18,40,45.0,-5.0
3,rule_execution_heavy,industryor_hard,20,20,11,55.0,11,20,55.0,0.0
4,rule_objective_heavy,mamo_easy,40,40,36,90.0,36,40,90.0,0.0
...,...,...,...,...,...,...,...,...,...,...
31,cap_revision_heavy,industryor_hard,20,20,10,50.0,11,20,55.0,-5.0
32,cap_semantic_numeric_heavy,mamo_easy,40,40,36,90.0,36,40,90.0,0.0
33,cap_semantic_numeric_heavy,mamo_complex,50,50,37,74.0,38,50,76.0,-2.0
34,cap_semantic_numeric_heavy,industryor_medium,40,40,15,37.5,18,40,45.0,-7.5


## Inspect the observations

The missing-final-result flag is distinct from an observed empty output. Inspect it directly, along with its analysis score:

In [8]:
reward = pd.read_csv(root / 'data/reward_results.csv')
missing = reward[~reward.final_result_available]
display(missing[['variant', 'dataset', 'problem_id', 'final_result_available', 'observed_correct', 'correct', 'output_state']])

,variant,dataset,problem_id,final_result_available,observed_correct,correct,output_state
740,sem_implementation_heavy,industryor_hard,36,False,NaN,False,missing_final_record
